# 📥 RAG Retrieval (FAISS)

This notebook handles the **expensive** part of the RAG evaluation pipeline:  
loading a pre-built FAISS index, running batch retrieval across strategies, and saving raw results to disk.

## Pipeline
1. **Load** the QA dataset (questions + ground-truth Wikipedia IDs)
2. **Load** the FAISS index from disk
3. **Retrieve** top-K documents for every question using each strategy
4. **Save** raw retrieval results as Parquet files
5. **Generate metadata** (decile boundaries — computed once per corpus)

## Strategies
- **Dense Vector Search** (`approximation` / `vector`): Semantic similarity via FAISS
- **BM25 Keyword Search** (`bm25`): Traditional BM25 (if a BM25 index was built alongside)

## Output
Results are saved to `{COLLECTION_ROOT}/{OUTPUT_NAME}/` as `results_{strategy}.parquet`.  
These files are consumed by `rag_evaluation_faiss.ipynb` for metric computation and visualization.

> **Note:** Run this notebook once (or when you change retrieval settings).  
> The evaluation notebook can be re-run cheaply on saved results.

## 1. Configuration

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv

from rag.faiss_rag_service import FaissRagService
from rag.utils import IndexingConfig
from config import DATA_DIR

load_dotenv()

# ============================================================================
# QA & Corpus Settings
# ============================================================================
OUTPUT_NAME = "nq_300_b"  # Must match the QA file name (without .parquet)

# Collection & Index (MUST match rag_indexing_faiss.ipynb)
COLLECTION_NAME = "wiki_100k_f"
COLLECTION_ROOT = Path(DATA_DIR) / COLLECTION_NAME
QUESTIONS_PATH = COLLECTION_ROOT / f"{OUTPUT_NAME}.parquet"
CORPUS_PATH = COLLECTION_ROOT / "wiki_corpus.parquet"
FAISS_INDEX_DIR = COLLECTION_ROOT / "faiss_index"  # Where the FAISS index was saved

# Embedding Configuration (MUST match indexing settings exactly!)
EMBEDDING_PROVIDER = "huggingface"
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Retrieval Configuration
STRATEGIES = ["approximation"]  # Strategies to evaluate (must have been indexed)
TOP_K = 10
K_VALUES_DETAILED = [1, 3, 5, 10]
MAX_QUESTIONS = None  # Set to limit for testing (e.g., 100), None for all

# Output Configuration
OUTPUT_FOLDER = OUTPUT_NAME
RESULTS_DIR = COLLECTION_ROOT / OUTPUT_FOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Config loaded")
print(f"  Collection: {COLLECTION_NAME}")
print(f"  FAISS index: {FAISS_INDEX_DIR}")
print(f"  Questions: {QUESTIONS_PATH}")
print(f"  Embedding: {EMBEDDING_PROVIDER} / {EMBEDDING_MODEL}")
print(f"  Top-K: {TOP_K}  |  K-values: {K_VALUES_DETAILED}")
print(f"  Strategies: {STRATEGIES}")
print(f"  Results dir: {RESULTS_DIR}")

✓ Config loaded
  Collection: wiki_100k_f
  FAISS index: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_100k_f/faiss_index
  Questions: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_100k_f/nq_300_b.parquet
  Embedding: huggingface / intfloat/multilingual-e5-small
  Top-K: 10  |  K-values: [1, 3, 5, 10]
  Strategies: ['approximation']
  Results dir: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_100k_f/nq_300_b


## 2. Load Questions

Load the QA dataset containing questions, ground-truth Wikipedia IDs, and popularity scores.

In [2]:
print("Loading questions...")
qa_df = pd.read_parquet(QUESTIONS_PATH)
qa_df = qa_df.dropna(subset=["question_text"])

# Normalize Wikipedia IDs to string for matching
qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(str).str.strip()

if MAX_QUESTIONS:
    qa_df = qa_df.sample(n=min(MAX_QUESTIONS, len(qa_df)), random_state=42)
    print(f"  Limited to {len(qa_df)} questions for testing")

print(f"✓ Loaded {len(qa_df):,} questions")
print(f"  Unique docs: {qa_df['wikipedia_id'].nunique():,}")
if "dataset" in qa_df.columns:
    print(f"  Datasets: {qa_df['dataset'].value_counts().to_dict()}")

print("\nSample:")
display(qa_df.head(3))

Loading questions...
✓ Loaded 799 questions
  Unique docs: 533
  Datasets: {'natural_questions': 799}

Sample:


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,pop_decile_unweighted,pop_decile_chunk_weighted,decile
0,-4574431015526060418,who played will munson on as the world turns,"[Jesse Soffer, ]",8376982,Will Munson,26.062500,5.018482e+06,natural_questions,3,1,1
1,847928453419463311,who sang please don't talk to the lifeguard,"[Diane Ray, ]",52678294,Please Don't Talk to the Lifeguard,64.833333,2.952597e+06,natural_questions,4,2,2
2,915941345973760946,who wrote little big town when someone stops l...,"[Hillary Lindsey, Chase McGill, Lori McKenna]",53144789,When Someone Stops Loving You,101.058333,2.663705e+06,natural_questions,5,2,2


## 3. Load FAISS Index

Load the pre-built FAISS index from disk.  
The embedding model and chunking parameters **must** match what was used during indexing.

In [3]:
print("Loading FAISS index...")

config = IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    embedding_provider=EMBEDDING_PROVIDER,
    embedding_model=EMBEDDING_MODEL,
    trust_remote_code=True,
    use_progress=False,
)

service = FaissRagService(
    config=config,
    strategy=STRATEGIES[0],  # Primary strategy; overridden per-query below
    distance_strategy="cosine",
)

service.load_index(FAISS_INDEX_DIR, progress_bar=True)

n_vectors = service.get_store().index.ntotal
print(f"✓ FAISS index loaded: {n_vectors:,} vectors")

if service._bm25_retriever is not None:
    n_bm25_docs = len(service._bm25_retriever.docs)
    print(f"✓ BM25 retriever ready: {n_bm25_docs:,} documents")
else:
    print("⚠  BM25 retriever not available")


Loading FAISS index...
✓ FAISS index loaded: 416,864 vectors
✓ BM25 retriever ready: 416,864 documents


## 4. Run Batch Retrieval

For each strategy, retrieve top-K documents for every question.  
Results are collected into DataFrames and saved as Parquet files.

In [ ]:
print("Running retrieval for all strategies...\n")

from helpers.decile_utils import COL_DECILE_UNWEIGHTED, COL_DECILE_CHUNK_WEIGHTED

retrieve_k = max(TOP_K, max(K_VALUES_DETAILED))
questions = qa_df["question_text"].tolist()

results_by_strategy = {}

for strategy in STRATEGIES:
    print(f"{'=' * 60}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'=' * 60}")

    # Skip if results already saved from a previous run
    output_path = RESULTS_DIR / f"results_{strategy}.parquet"
    if output_path.exists():
        print(f"⏭  Skipping {strategy} — results already exist at {output_path}\n")
        results_by_strategy[strategy] = pd.read_parquet(output_path)
        continue

    all_results = service.batch_retrieve(
        questions=questions,
        top_k=retrieve_k,
        strategy=strategy,
    )

    # ── Vectorised result processing ─────────────────────────────────
    rows = []
    qa_records = qa_df.to_dict("records")

    for qr, retrieved_docs in zip(qa_records, all_results):
        retrieved_ids = []
        retrieved_scores = []
        retrieved_pops = []

        for doc, score in retrieved_docs:
            raw_id = doc.metadata.get("wikipedia_id", doc.metadata.get("id", ""))
            doc_id = str(int(float(raw_id))) if raw_id not in (None, "") else ""
            retrieved_ids.append(doc_id)
            retrieved_scores.append(score)
            retrieved_pops.append(doc.metadata.get("popularity_avg"))

        rows.append({
            "question": qr["question_text"],
            "wikipedia_id": str(qr["wikipedia_id"]).strip(),
            "wikipedia_title": qr.get("wikipedia_title"),
            "popularity_avg": qr.get("popularity_avg"),
            "dataset": qr.get("dataset"),
            COL_DECILE_UNWEIGHTED: qr.get(COL_DECILE_UNWEIGHTED, -1),
            COL_DECILE_CHUNK_WEIGHTED: qr.get(COL_DECILE_CHUNK_WEIGHTED, -1),
            "decile": qr.get("decile", -1),
            "topk_ids": retrieved_ids,
            "topk_scores": retrieved_scores,
            "topk_popularities": retrieved_pops,
        })

    results_df = pd.DataFrame(rows)
    results_by_strategy[strategy] = results_df

    # Save immediately after this strategy finishes
    results_df.to_parquet(output_path)
    print(f"  💾 Saved {strategy} → {output_path} ({len(results_df):,} rows)\n")

ALL_STRATEGIES = list(results_by_strategy.keys())
print(f"✅ All retrieval complete: {', '.join(ALL_STRATEGIES)}")


Running retrieval for all strategies...

Strategy: APPROXIMATION


Retrieving (approximation):   0%|          | 0/799 [00:00<?, ?it/s]

✓ Retrieved 799 queries

✅ All retrieval complete: approximation


## 5. Save Raw Results

Save retrieval results as Parquet files — one per strategy.

In [ ]:
print("Saving retrieval results...\n")

for strategy in ALL_STRATEGIES:
    output_path = RESULTS_DIR / f"results_{strategy}.parquet"
    results_by_strategy[strategy].to_parquet(output_path)
    print(f"  ✓ {strategy}: {output_path} ({len(results_by_strategy[strategy]):,} rows)")

print(f"\n✅ All results saved to: {RESULTS_DIR}")

Saving retrieval results...



,question,wikipedia_id,wikipedia_title,popularity_avg,dataset,pop_decile_unweighted,pop_decile_chunk_weighted,decile,topk_ids,topk_scores,topk_popularities
0,who played will munson on as the world turns,8376982,Will Munson,26.062500,natural_questions,3,1,1,"[8376982, 1422165, 2741720, 1422165, 26175492,...","[0.25514162, 0.32636747, 0.3554492, 0.35909683...","[26.0625, 239.02083333333334, 384.125, 239.020..."
1,who sang please don't talk to the lifeguard,52678294,Please Don't Talk to the Lifeguard,64.833333,natural_questions,4,2,2,"[52678294, 43286612, 6512501, 6512501, 4568593...","[0.20003426, 0.31397128, 0.35353163, 0.3758341...","[64.83333333333334, 58.58333333333333, 6984.72..."
2,who wrote little big town when someone stops l...,53144789,When Someone Stops Loving You,101.058333,natural_questions,5,2,2,"[53144789, 34829017, 39551094, 273373, 4043289...","[0.21180539, 0.3518882, 0.36986077, 0.3740818,...","[101.05833333333334, 228.79166666666669, 206.0..."


  ✓ approximation: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_100k_f/nq_300_b/results_approximation.parquet (799 rows)

✅ All results saved to: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_100k_f/nq_300_b


## 6. Generate Metadata (once per corpus)

Calculates decile boundaries (unweighted + chunk-weighted) and saves configuration metadata.  
If `metadata.json` already exists (e.g. from a previous run with the same corpus),
it is **reused** — only strategy/question-specific fields are updated.

In [ ]:
import json
from helpers.decile_utils import (
    compute_corpus_boundaries,
    boundaries_to_metadata,
    print_boundaries,
    _META_KEY_UNWEIGHTED,
)

metadata_path = RESULTS_DIR / "metadata.json"

# ── Load existing metadata if present ────────────────────────────────
existing_meta = {}
if metadata_path.exists():
    with open(metadata_path) as f:
        existing_meta = json.load(f)

# ── Decide whether corpus boundaries need (re-)computation ──────────
need_boundaries = _META_KEY_UNWEIGHTED not in existing_meta

if need_boundaries and CORPUS_PATH.exists():
    print("Computing decile boundaries from corpus (first time only)...\n")
    boundaries_uw, boundaries_cw, stats, _ = compute_corpus_boundaries(
        corpus_path=CORPUS_PATH,
        batch_size=100_000,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    print(f"\n✓ Boundaries computed")
    print(f"  Unique docs: {stats['unique_documents_with_popularity']:,}")
    print(f"  Total chunks: {stats['total_chunks_after_splitting']:,}")
    print_boundaries(boundaries_uw, boundaries_cw)

    boundary_meta = boundaries_to_metadata(boundaries_uw, boundaries_cw, stats, CHUNK_SIZE, CHUNK_OVERLAP)
elif need_boundaries:
    print(f"⚠️  Corpus not found at {CORPUS_PATH} — metadata will lack boundaries")
    boundary_meta = {}
else:
    print("✓ Decile boundaries already in metadata — skipping recomputation")
    boundary_meta = {}  # Don't overwrite existing boundaries

# ── Merge run-specific fields into metadata ──────────────────────────
run_meta = {
    "collection_name": COLLECTION_NAME,
    "output_name": OUTPUT_NAME,
    "backend": "faiss",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_provider": EMBEDDING_PROVIDER,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "strategies": ALL_STRATEGIES,
    "top_k": TOP_K,
    "k_values_detailed": K_VALUES_DETAILED,
    "num_questions": len(qa_df),
    "corpus_path": str(CORPUS_PATH),
    "faiss_index_dir": str(FAISS_INDEX_DIR),
}

metadata = {**existing_meta, **boundary_meta, **run_meta}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Metadata saved to: {metadata_path}")
print(f"\nNext: Open rag_evaluation_faiss.ipynb to compute metrics and generate visualizations.")

✓ Decile boundaries already in metadata — skipping recomputation

✅ Metadata saved to: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_100k_f/nq_300_b/metadata.json

Next: Open rag_evaluation_faiss.ipynb to compute metrics and generate visualizations.
